* Since Image inputs (RGB) have a huge number of input features, it is impossible to use traditional NN to train.
* We want to train a network to be able to detect pattern, regardless of where it appears.

**CNN** is NN that use Conv layers, which consist of a set of filters in form of 2d matrices of numbers
1. Overlaying the filter on top of the image at some location
2. Performing element-wise multiplication between the values in the filter and their corresponding values in the images
3. Summing up all the element-wise products. This sum is the output value for the destination pixel in the output image
4. Repeating for all locations.

<!> Library like **PyTorch** actually implements a cross correlation method instead of convolution. This is because cross-correltion will give a better performance since it avoids the additional step of flipping the filters to perform the convolutions.

In [13]:
import numpy as np

In [14]:
def cross_correlation_2d(signal, kernel): # or input and filter
    '''
    2D cross-correlation with output size shrinks by (kernel - 1)
    '''
    img_h, img_w = signal.shape
    ker_h, ker_w = kernel.shape

    # use valid padding
    out_h = img_h - ker_h + 1
    out_w = img_w - ker_w + 1

    output = np.zeros((out_h, out_w))

    for i in range(out_h): # vertical slide
        for j in range(out_w): # horizontal slide
            # get the region matchs the size of kernel
            roi = signal[i : i + ker_h, j : j + ker_w]
            
            # element-wise multiplication and sum the product
            output[i, j] = np.sum(roi * kernel)

    return output

def convolution_2d(signal, kernel):
    '''
    Flip the kernel, then performs corss-correlation
    '''
    # flip both up -> down, and left -> right
    kernel_flipped = np.flip(kernel, axis=(0,1))

    return cross_correlation_2d(signal, kernel_flipped)

In [15]:
image = np.array([
    [10, 10,  0,  0],
    [10, 10,  0,  0],
    [ 0,  0, 10, 10],
    [ 0,  0, 10, 10]
])

# A generic "Edge Detection" Kernel (Vertical edge)
kernel = np.array([
    [1, 0],
    [0, 2]
])

# 1. Run Correlation (No Flip)
corr_result = cross_correlation_2d(image, kernel)

# 2. Run Convolution (Flip)
conv_result = convolution_2d(image, kernel)

print("Original Kernel:\n", kernel)
print("-" * 20)
print("Cross-Correlation Result:\n", corr_result)
print("-" * 20)
print("Convolution Result:\n", conv_result)

Original Kernel:
 [[1 0]
 [0 2]]
--------------------
Cross-Correlation Result:
 [[30. 10.  0.]
 [10. 30. 20.]
 [ 0. 20. 30.]]
--------------------
Convolution Result:
 [[30. 20.  0.]
 [20. 30. 10.]
 [ 0. 10. 30.]]


In [16]:
# vertical Sobel filter to filter out vertical edge
ver_sobel_filter = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
])

# horizontal Sobel filter for hortizontal edge
hor_sobel_filter = np.array([
    [1, 2, 1],
    [0, 0, 0],
    [-1, -2, -1]
])

Oftern times, we would prefer to have the output image to be the same size as the input image. To achieve this, we add paddings to compromise the shrinks.

In [17]:
# implement CNN for MNIST, with 8 layers of 3 x 3 filter, we will produce 8 layers. So from the original 28 x 28 inputs, we can convolute it to 26 x 26 x 8, and reduce from 784 weights to 72 weights (3 x 3 x 8)
class Conv3x3:
    def __init__(self, num_filters):
        self.num_filters = num_filters
        self.filters = np.random.randn(num_filters, 3, 3) / 9 # reduce variance

    def iterate_regions(self, image):
        # with valid padding
        h, w = image.shape
        
        for i in range(h - 2):
            for j in range(w - 2):
                img_reg = image[i : (i + 3), j : (j + 3)]
                yield img_reg, i, j

    def forward(self, input):
        # return a 3d array (h - 2, w - 2, num_filters)
        h, w = input.shape
        self.last_input = input
        output = np.zeros((h - 2, w -2, self.num_filters))

        for img_reg, i, j in self.iterate_regions(input):
            # np.sum return 1d array with len = num_filters
            output[i,j] = np.sum(img_reg * self.filters, (1,2))

        return output
    
    def backprop(self, d_L_d_out, learn_rate):
        d_L_d_filters = np.zeros(self.filters.shape)
        for img_region, i, j in self.iterate_regions(self.last_input):
            for f in range(self.num_filters):
                d_L_d_filters[f] += d_L_d_out[i, j, f] * img_region

        
        self.filters -= learn_rate * d_L_d_filters


In [18]:
import mnist
import scipy.misc as scm
mnist.datasets_url = 'https://storage.googleapis.com/cvdf-datasets/mnist/'

train_imgs = mnist.train_images()
train_labels = mnist.train_labels()


/tmp/ipykernel_474377/367528345.py:2: DeprecationWarning: scipy.misc is deprecated and will be removed in 2.0.0
  import scipy.misc as scm


In [19]:
conv = Conv3x3(8)

output = conv.forward(train_imgs[0])
print(output.shape)

(26, 26, 8)


Neighboring pixels in images tend to have similar values, so conv layers will also produce similar values for neighboring pixels in outputs.

-> Much of information outputed by conv layer is redundant.

-> solve by pooling layers, usually perform simple operation: `max`, `min` or `average`. After going through pooling layer, the input size is reduced.

In [20]:
class MaxPool2:
    # max pooling layer using pool size of 2 (2x2 -> 1)
    def iterate_regions(self, image): 
        h, w, _ = image.shape
        self.last_input = image
        new_h = h // 2
        new_w = w // 2

        for i in range(new_h):
            for j in range(new_w):
                img_region = image[(i * 2):(i * 2 + 2), (j * 2):(j * 2 + 2)]
                yield img_region, i, j

    def forward(self, input):
        h, w, num_filters = input.shape
        output = np.zeros((h//2, w//2, num_filters))
        for img_region, i, j in self.iterate_regions(input):
            output[i,j] = np.amax(img_region, axis=(0, 1))

        return output 
    
    # still need to implement backprop in order to calculate gradients
    def backprop(self, d_L_d_out):
        d_L_d_input = np.zeros(self.last_input.shape)
        
        for img_region, i, j in self.iterate_regions(self.last_input):
            h, w, f = img_region.shape
            amax = np.amax(img_region, axis=(0, 1))

            for i2 in range(h):
                for j2 in range(w):
                    for f2 in range(f):
                        if img_region[i2, j2, f2] == amax[f2]:
                            d_L_d_input[i * 2 + i2, j * 2 + j2, f2] = d_L_d_out[i, j, f2]

        return d_L_d_input


In [21]:
pool = MaxPool2()

# from output 26 x 26 x 8
output = pool.forward(output)
print(output.shape)

(13, 13, 8)


Now we use Softmax as activation function and final layer for multiclass classifciation problem.

$$
s(x_i) = \frac{e^{x_i}}{\sum^n_{j=1} e^{x_j}}
$$

In [22]:
class Softmax:
    def __init__(self, input_len, nodes):
        self.weights = np.random.randn(input_len, nodes) / input_len # random weight and biases -> not good result
        self.biases = np.zeros(nodes)

    def forward(self, input):
        # output 1d np array 
        # input can be any array with any dimensions

        self.last_input_shape = input.shape # cache input_shape
        input = input.flatten()
        self.last_input = input # cache input

        input_len, nodes = self.weights.shape
        totals = np.dot(input, self.weights) + self.biases
        exp = np.exp(totals)
        
        self.last_totals = totals # cache totals

        return exp / np.sum(exp, axis=0)
    
    def backprop(self, d_L_d_out, learn_rate):
        '''
        - d_L_d_out is loss gradient for this layer output
        '''
        for i, gradient in enumerate(d_L_d_out):
            if gradient == 0:
                continue

            t_exp = np.exp(self.last_totals)

            S = np.sum(t_exp)

            # gradients of out[i] against totals
            d_out_d_t = -t_exp[i] * t_exp / (S ** 2)
            d_out_d_t[i] = t_exp[i] * (S - t_exp[i]) / (S ** 2)

            # grad of totals against w, b, inputs
            d_t_d_w = self.last_input
            d_t_d_b = 1
            d_t_d_inputs = self.weights

            # grad of loss against totals
            d_L_d_t = gradient * d_out_d_t

            # grad of loss against w, b, inputs
            # since d_t_d_w and d_l_d_t are 1d arrays, so we need to convert it to (input_len, 1) and (nodes, 1) for easier transpose, the result will have shape (input_len, nodes) which is the same as self.weights
            d_L_d_w = d_t_d_w[np.newaxis].T @ d_L_d_t[np.newaxis]
            d_L_d_b = d_L_d_t * d_t_d_b
            d_L_d_inputs = d_t_d_inputs @ d_L_d_t

            # Update weights / biases
            self.weights -= learn_rate * d_L_d_w
            self.biases -= learn_rate * d_L_d_b
            return d_L_d_inputs.reshape(self.last_input_shape)

In [23]:
train_imgs = mnist.train_images()[:1000]
train_labels = mnist.train_labels()[:1000]
test_imgs = mnist.test_images()[:1000]
test_labels = mnist.test_labels()[:1000]

conv = Conv3x3(8) # 28 x 28 x 1 -> 26 x 26 x 8
pool = MaxPool2() # 26 x 26 x 8 -> 13 x 13 x 8
softmax = Softmax(13 * 13 * 8, 10) # 13 x 13 x 8 -> 10

def forward(image, label):
    '''
    - image is a 2d numpy array
    - label is a digit
    '''
    output = conv.forward((image/ 255) - 0.5)
    output = pool.forward(output)
    output = softmax.forward(output)

    # calculate cross-entropy loss
    loss = -np.log(output[label])
    acc = 1 if np.argmax(output) == label else 0
    return output, loss, acc


In [24]:
def train(im, label, lr=0.005):
    '''
    - im is a 2d numpy array
    - label is a digit
    - lr = learning rate
    '''
    out, loss, acc = forward(im, label)

    grad = np.zeros(10)
    grad[label] = -1/out[label]

    # backprop
    grad = softmax.backprop(grad, lr)
    # TODO: backprop MaxPool2 and Conv3x3 layer
    grad = pool.backprop(grad)
    grad = conv.backprop(grad, lr)

    return loss, acc

In [25]:
print('MNIST CNN initialized!')

# train CNN

for epoch in range(5):
  print('--- Epoch %d ---' % (epoch + 1))

  # Shuffle the training data
  permutation = np.random.permutation(len(train_imgs))
  train_imgs = train_imgs[permutation]
  train_labels = train_labels[permutation]

  # train
  loss = 0
  num_correct = 0
  for i, (im, label) in enumerate(zip(train_imgs, train_labels)):
    if i % 100 == 99:
      print(
        '[Step %d] Past 100 steps: Average Loss %.3f | Accuracy: %d%%' %
        (i + 1, loss / 100, num_correct)
      )
      loss = 0
      num_correct = 0
    
    l, acc = train(im, label)
    loss += 1
    num_correct += acc

MNIST CNN initialized!
--- Epoch 1 ---
[Step 100] Past 100 steps: Average Loss 0.990 | Accuracy: 13%
[Step 200] Past 100 steps: Average Loss 1.000 | Accuracy: 37%
[Step 300] Past 100 steps: Average Loss 1.000 | Accuracy: 51%
[Step 400] Past 100 steps: Average Loss 1.000 | Accuracy: 60%
[Step 500] Past 100 steps: Average Loss 1.000 | Accuracy: 71%
[Step 600] Past 100 steps: Average Loss 1.000 | Accuracy: 68%
[Step 700] Past 100 steps: Average Loss 1.000 | Accuracy: 76%
[Step 800] Past 100 steps: Average Loss 1.000 | Accuracy: 78%
[Step 900] Past 100 steps: Average Loss 1.000 | Accuracy: 84%
[Step 1000] Past 100 steps: Average Loss 1.000 | Accuracy: 76%
--- Epoch 2 ---
[Step 100] Past 100 steps: Average Loss 0.990 | Accuracy: 80%
[Step 200] Past 100 steps: Average Loss 1.000 | Accuracy: 85%
[Step 300] Past 100 steps: Average Loss 1.000 | Accuracy: 84%
[Step 400] Past 100 steps: Average Loss 1.000 | Accuracy: 92%
[Step 500] Past 100 steps: Average Loss 1.000 | Accuracy: 87%
[Step 600] Pas

In [26]:
# test CNN
print('\n--- Testing the CNN ---')
loss = 0
num_correct = 0
for im, label in zip(test_imgs, test_labels):
  _, l, acc = forward(im, label)
  loss += l
  num_correct += acc

num_tests = len(test_imgs)
print('Test Loss:', loss / num_tests)
print('Test Accuracy:', num_correct / num_tests)


--- Testing the CNN ---
Test Loss: 0.47205867760624814
Test Accuracy: 0.86


As we can see, the result is pretty bad, since we haven't implemented backpropagated and updates on weights.
* Forward phase: input is passed completely through the network.
    - Each layer will cache data, which will need for the backward phase
* Backward phase: gradients are backpropagated and weight are updated.
    - Each layer will receive a gradient (derivatives on loss with outputs) and return a gradient (derivatives on loss with inputs)

$$
\text{out}_s(c) = \frac{e^{t_c}}{\sum _i e^{t_i}} = \frac{e^{t_c}}{S} = e^{t_c} S^{-1}
$$

where c is correct class, $t_i$ is the total for class i. And with class k $\neq$ c, we have

\begin{gather*}
\frac{\delta \textbf{out}_s (c)}{\delta t_k} = \frac{\delta \textbf{out}_s (c)}{\delta S} \frac{\delta S}{\delta t_k} = \frac{-e^{t_c} e^{t_k}}{S^2} \\
\frac{\delta \text{out}_s(c)}{\delta t_c} = \frac{e^{t_c}(S - e^{t_c})}{S^2}
\end{gather*}


We have total $t = w * \text{input} + b$. So we need to derive 3 more results:
\begin{gather*}
\frac{\delta t}{\delta w} = input \\
\frac{\delta t}{\delta b} = 1 \\
\frac{\delta t}{\delta \text{input}} = w
\end{gather*}

Put all together:
\begin{gather*}
    \frac{\partial L}{\partial w} = \frac{\partial L}{\partial out} * \frac{\partial out}{\partial t} * \frac{\partial t}{\partial w} \\[2ex]
    \frac{\partial L}{\partial b} = \frac{\partial L}{\partial out} * \frac{\partial out}{\partial t} * \frac{\partial t}{\partial b} \\[2ex]
    \frac{\partial L}{\partial input} = \frac{\partial L}{\partial out} * \frac{\partial out}{\partial t} * \frac{\partial t}{\partial input}
\end{gather*}

Back propagation for Convolution Layer
We assume that Conv3x3 is used as the first layer in our network. If we need to use Conv3x3 multiple times, we would have to make it accept 3d array input.

$$
\frac{\delta L}{\delta \text{filter} (x,y)} = \sum _i \sum _j \frac{\delta L}{\delta \text{out}(i,j)} * \frac{\delta \text{out}(i, j)}{\delta \text{filter}(x,y)}
$$